# Look at Weird HEC RAS Stuff

There is a wave bouncing off the downstream boundary at a certain point and then weird low concentration things are happening. This notebook is intended to look at what's going on with that. 

## Set Up

In [28]:
import clearwater_riverine as cwr
from pathlib import Path
import numpy as np
import pandas as pd

In [8]:
#point to config
network_path =Path('../data_temp/mississippi_model')

#point to config
config_file = network_path / 'demo_config_20s_no_diffusion.yml'
print(config_file.exists())

True


In [9]:
# run small model - skip warmup 
# warmup_time = 20 * 24
start_index = 2100 # int((warmup_time * 60) / 10) 
# hours_to_run = 10 * 24
end_index = 5000   # start_index + int(hours_to_run * 60 / 10) 

In [10]:
print(start_index, end_index)

2100 5000


## Model Run

In [4]:
%%time
transport_model = cwr.ClearwaterRiverine(
    config_filepath=config_file,
    verbose=True,
    datetime_range= (start_index, end_index)
)

Populating Model Mesh...
'Cell Hydraulic Depth' not found in hdf file; skip reading it. 
'Cell Velocity - Velocity X' not found in hdf file; skip reading it. 
'Cell Velocity - Velocity Y' not found in hdf file; skip reading it. 
Cell velocities X and Y not found in hdf file; skip calculating velocity magnitude
Extra boundary faces identified for Amite South.
Removing erroneous boundaries {1, 180227, 6, 9, 180234, 16, 18, 23, 25, 180249, 188445, 30, 180255, 180256, 31, 188450, 34, 39, 41, 180267, 43, 180269, 180272, 180273, 48, 180275, 188468, 180277, 52, 57, 180285, 62, 180291, 180292, 147529, 180298, 78, 82, 85, 172123, 147548, 96, 147552, 98, 180324, 147557, 180326, 103, 147558, 8297, 180333, 147566, 147567, 172141, 180338, 180341, 147573, 172150, 147578, 147580, 163965, 8318, 127, 147582, 147585, 163969, 8323, 180356, 147590, 180360, 180361, 106635, 163979, 147597, 147599, 172179, 8342, 153, 172185, 180379, 180380, 155, 147609, 172191, 180385, 32933, 180389, 172198, 180392, 172200, 

In [5]:
%%time
for t in range(len(transport_model.mesh.time) - 1):
    transport_model.update()

CPU times: total: 2h 39min 20s
Wall time: 51min 20s


## Plot

In [12]:
transport_model.quick_plot(
    clim=(99.5,100.5)
)


C:\Users\sjordan\OneDrive - LimnoTech\Documents\GitHub\ClearWater-riverine\src\clearwater_riverine\transport.py:560: UserWarning: No constituent name defined. Plotting tracer.
  warnings.warn(


:DynamicMap   [Time]
   :Scatter   [x,y]   (x,y,tracer,nface)

## Look at Volume and Face Flow

### Define Functions

In [13]:
def get_cell_edges(mesh, cell_id):
    """
    Returns the indices of edges connected to a given cell.
    """
    mask = np.any(mesh.edge_face_connectivity == cell_id, axis=1)
    edge_indices = np.where(mask)[0]
    return edge_indices

In [124]:
def check_volume_difference(df, cell, print_output, threshold=0.1):
    """
    Checks what percentage of rows have a percent_difference greater than the given threshold.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing the column 'percent_difference'
    threshold : float
        Threshold in percent (e.g., 0.1 means 0.1%)

    """
    # Ensure percent_difference exists
    if "percent_difference" not in df.columns:
        raise ValueError("DataFrame must contain a 'percent_difference' column.")
    
    # Take absolute value for comparison
    mask = df["percent_difference"].abs() > threshold
    
    # Calculate percentage of rows exceeding threshold
    total_rows = len(df)
    if total_rows == 0:
        return 0.0

    percent_exceeding = (mask.sum() / total_rows) * 100

    # Print the result
    if print_output:
        print(f"Cell {cell}: {percent_exceeding:.2f}% of rows exceed the {threshold}% threshold.")
    
    return percent_exceeding

In [125]:
def build_cell_dataframe(cell, time_slice=None, threshold=0.1, print_output=False):
    """
    Create a DataFrame of volume and face flows for a given cell
    over a specified time range or the full time series.

    Parameters
    ----------
    cell : int
        Index of the cell to extract data for.
    time_slice : int, slice, list, or None
        If None, includes all timesteps.
        Otherwise, pass a single timestep index or a slice.
     threshold : float
            Threshold in percent (e.g., 0.1 means 0.1%)
    
    """
    # --- Determine which timesteps to include
    if time_slice is None:
        time_idx = slice(None)  # full range
    else:
        time_idx = time_slice

    # --- Identify edges connected to the cell
    edges = get_cell_edges(transport_model.mesh, cell)

    # Get connectivity for these edges
    pairs = transport_model.mesh.edge_face_connectivity[edges]

    # Determine the sign: +1 if cell is first in pair, -1 if second
    sign = np.where(pairs[:, 0] == cell, 1, -1)  # shape (num_edges,)

    # --- Extract face_flow for all these edges at once
    face_flow = transport_model.mesh.face_flow.isel(nedge=edges, time=time_idx).values * sign  # shape (num_edges, tim

    # --- Pad face_flow to nmax_face
    num_edges = face_flow.shape[1]
    nmax_face = len(transport_model.mesh.nmax_face)

    if num_edges < nmax_face:
        pad_width = nmax_face - num_edges
        face_flow = np.pad(face_flow, ((0, 0), (0, pad_width)), constant_values=np.nan)


    # --- Get volume for the cell
    volume = transport_model.mesh.volume.isel(nface=cell, time=time_idx).values

    # --- Get timestep values
    time_vals = transport_model.mesh.time.isel(time=time_idx).values

    # --- Handle dt (could be array or scalar)
    dt_vals = transport_model.mesh.dt.isel(time=time_idx).values


    # --- Build DataFrame
    data = {
        "time": time_vals,
        "volume": volume,
    }

    # Add face_flow columns
    for i in range(nmax_face):
        data[f"face_flow_{i}"] = face_flow[:, i]

    # Create DataFrame
    df = pd.DataFrame(data)

    # --- Compute sum of face flows
    face_flow_cols = [f"face_flow_{i}" for i in range(nmax_face)]
    df["sum_face_flows"] = df[face_flow_cols].sum(axis=1, skipna=True)

    # --- Compute volume change (difference between timesteps)
    df["volume_change_volume"] = df["volume"].diff()

    # --- Add timestep change column
    df["change_in_time"] = dt_vals

    # --- Compute total volume change
    df["volume_change_face_flow"] = df["sum_face_flows"] * df["change_in_time"]

    # --- add cell
    df["cell"] = cell

    # --- Differences
    df['total_difference'] = df['volume_change_volume'] + df['volume_change_face_flow']
    # Calculate percent difference
    # (plus signs due to different signs)
    df["percent_difference"] = (
        (df["volume_change_face_flow"] + df["volume_change_volume"]) / df["volume_change_volume"]) * 100

    diff = check_volume_difference(df, cell, print_output, threshold=threshold)

    df_diff = pd.DataFrame(
        {
            'cell': [cell],
            'diff': [diff]
        }
    )
    

    return df, df_diff


### Test Functions

In [127]:
#68440 is the cell next to the downstream boundary where weird thigns start happening:

results, results_df = build_cell_dataframe(68440, print_output=True, threshold=0.5)

Cell 68440: 39.43% of rows exceed the 0.5% threshold.


### Examine Problem-Range

This is the time range that first flagged the problem for me. See how the second and third row of volume_change_face_flow equal the third flow of volume_change_volume. 

In [98]:
results[(results.time >= "2018-03-29T16:59:40.000000000") & (results.time <= "2018-03-29T17:03:00.000000000")]

,time,volume,face_flow_0,face_flow_1,face_flow_2,face_flow_3,face_flow_4,face_flow_5,face_flow_6,face_flow_7,sum_face_flows,volume_change_volume,change_in_time,volume_change_face_flow,cell,total_difference,percent_difference
779,2018-03-29 16:59:40,12132.893555,-0.041642,2.214015,-0.750326,0.0,-1.010374,NaN,NaN,NaN,0.411672,-8.258789,20.0,8.233449,68440,-0.025340,0.306827
780,2018-03-29 17:00:00,12124.660156,-0.042955,2.210290,-0.748474,0.0,-1.008385,NaN,NaN,NaN,0.410477,-8.233398,20.0,8.209531,68440,-0.023867,0.289886
781,2018-03-29 17:00:20,12108.624023,-0.044631,2.188760,-0.746548,0.0,-1.006248,NaN,NaN,NaN,0.391333,-16.036133,20.0,7.826665,68440,-8.209468,51.193566
782,2018-03-29 17:00:40,12101.720703,-0.047548,2.140442,-0.744292,0.0,-1.003398,NaN,NaN,NaN,0.345204,-6.903320,20.0,6.904071,68440,0.000751,-0.010874
783,2018-03-29 17:01:00,12096.330078,-0.053005,2.062839,-0.741320,0.0,-0.998983,NaN,NaN,NaN,0.269531,-5.390625,20.0,5.390615,68440,-0.000010,0.000193
784,2018-03-29 17:01:20,12092.994141,-0.062553,1.958426,-0.737151,0.0,-0.991973,NaN,NaN,NaN,0.166749,-3.335938,20.0,3.334985,68440,-0.000952,0.028546
785,2018-03-29 17:01:40,12092.176758,-0.077847,1.831276,-0.731253,0.0,-0.981275,NaN,NaN,NaN,0.040902,-0.817383,20.0,0.818036,68440,0.000654,-0.079955
786,2018-03-29 17:02:00,12094.270508,-0.100531,1.684712,-0.723073,0.0,-0.965823,NaN,NaN,NaN,-0.104716,2.093750,20.0,-2.094311,68440,-0.000561,-0.026785
787,2018-03-29 17:02:20,12099.620117,-0.132125,1.521344,-0.712055,0.0,-0.944639,NaN,NaN,NaN,-0.267476,5.349609,20.0,-5.349529,68440,0.000081,0.001506
788,2018-03-29 17:02:40,12108.527344,-0.173932,1.343096,-0.697646,0.0,-0.916873,NaN,NaN,NaN,-0.445355,8.907227,20.0,-8.907095,68440,0.000132,0.001478


Look to see the few timesteps before:

In [100]:
results[(results.time >= "2018-03-29T16:55:40.000000000") & (results.time <= "2018-03-29T17:03:00.000000000")]

,time,volume,face_flow_0,face_flow_1,face_flow_2,face_flow_3,face_flow_4,face_flow_5,face_flow_6,face_flow_7,sum_face_flows,volume_change_volume,change_in_time,volume_change_face_flow,cell,total_difference,percent_difference
767,2018-03-29 16:55:40,12233.654297,-0.026543,2.268842,-0.773944,0.0,-1.041749,NaN,NaN,NaN,0.426606,-8.554688,20.0,8.532116,68440,-0.022572,0.263855
768,2018-03-29 16:56:00,12225.122070,-0.027656,2.263708,-0.771958,0.0,-1.038644,NaN,NaN,NaN,0.425450,-8.532227,20.0,8.508996,68440,-0.023230,0.272263
769,2018-03-29 16:56:20,12216.613281,-0.028809,2.258650,-0.769963,0.0,-1.035619,NaN,NaN,NaN,0.424258,-8.508789,20.0,8.485164,68440,-0.023625,0.277657
770,2018-03-29 16:56:40,12208.127930,-0.029999,2.253677,-0.767964,0.0,-1.032678,NaN,NaN,NaN,0.423036,-8.485352,20.0,8.460718,68440,-0.024633,0.290305
771,2018-03-29 16:57:00,12199.666992,-0.031221,2.248797,-0.765963,0.0,-1.029824,NaN,NaN,NaN,0.421790,-8.460938,20.0,8.435797,68440,-0.025141,0.297139
772,2018-03-29 16:57:20,12191.231445,-0.032472,2.244020,-0.763964,0.0,-1.027058,NaN,NaN,NaN,0.420526,-8.435547,20.0,8.410517,68440,-0.025030,0.296723
773,2018-03-29 16:57:40,12182.821289,-0.033747,2.239351,-0.761970,0.0,-1.024386,NaN,NaN,NaN,0.419249,-8.410156,20.0,8.384974,68440,-0.025183,0.299430
774,2018-03-29 16:58:00,12174.435547,-0.035042,2.234799,-0.759984,0.0,-1.021807,NaN,NaN,NaN,0.417966,-8.385742,20.0,8.359321,68440,-0.026421,0.315073
775,2018-03-29 16:58:20,12166.076172,-0.036351,2.230371,-0.758012,0.0,-1.019324,NaN,NaN,NaN,0.416684,-8.359375,20.0,8.333671,68440,-0.025704,0.307486
776,2018-03-29 16:58:40,12157.743164,-0.037670,2.226072,-0.756056,0.0,-1.016939,NaN,NaN,NaN,0.415407,-8.333008,20.0,8.308146,68440,-0.024862,0.298353


Error is high before that spike as well. Not quite as high, but still pretty high. Before the spike in error, the volume_change_volume appears to correspodn to the volume_change_face_flow in the PRIOR timestep!

Then, at index 780 abnd 781, the flow across the face correspnoding to the large volume change is spread over two timesteps. Then, the volume calculated based on flow across face corresponds to the change in volume calcualted at each timestep.

Weird!

### Look at very large % Diffs

In [94]:
results[results.percent_difference > 50]

,time,volume,face_flow_0,face_flow_1,face_flow_2,face_flow_3,face_flow_4,face_flow_5,face_flow_6,face_flow_7,sum_face_flows,volume_change_volume,change_in_time,volume_change_face_flow,cell,total_difference,percent_difference
254,2018-03-29 14:04:40,12441.926758,-0.069796,1.974462,-0.865393,0.0,-1.025364,NaN,NaN,NaN,0.013909,-1.403320,20.0,0.278179,68440,-1.125142,80.177102
255,2018-03-29 14:05:00,12441.648438,-0.096062,1.910458,-0.850693,0.0,-1.002681,NaN,NaN,NaN,-0.038978,-0.278320,20.0,-0.779557,68440,-1.057877,380.093491
602,2018-03-29 16:00:40,13534.704102,-0.125535,1.813923,-0.633101,0.0,-1.052143,NaN,NaN,NaN,0.003144,0.096680,20.0,0.062880,68440,0.159559,165.039124
781,2018-03-29 17:00:20,12108.624023,-0.044631,2.188760,-0.746548,0.0,-1.006248,NaN,NaN,NaN,0.391333,-16.036133,20.0,7.826665,68440,-8.209468,51.193566
977,2018-03-29 18:05:40,20458.541016,-1.324838,-3.244766,2.662578,0.0,1.885745,NaN,NaN,NaN,-0.021282,1.548828,20.0,-0.425642,68440,1.123187,72.518478
978,2018-03-29 18:06:00,20458.966797,-1.259158,-3.091492,2.584409,0.0,1.783805,NaN,NaN,NaN,0.017564,0.425781,20.0,0.351288,68440,0.777070,182.504412
1321,2018-03-29 20:00:20,13758.312500,1.439959,2.484950,-1.608919,0.0,-0.643261,NaN,NaN,NaN,1.672730,0.000000,20.0,33.454591,68440,33.454591,inf
1515,2018-03-29 21:05:00,12585.679688,0.066119,1.367704,-0.720542,0.0,-0.710389,NaN,NaN,NaN,0.002892,-0.556641,20.0,0.057844,68440,-0.498796,89.608336
1516,2018-03-29 21:05:20,12585.622070,0.049991,1.334335,-0.710705,0.0,-0.693292,NaN,NaN,NaN,-0.019671,-0.057617,20.0,-0.393415,68440,-0.451033,782.808957
1869,2018-03-29 23:03:00,14611.133789,-0.555229,0.139783,0.438505,0.0,-0.085498,NaN,NaN,NaN,-0.062439,2.594727,20.0,-1.248770,68440,1.345956,51.872763


## Run Function for all Cells 

Set threshold to flag any cells that are > 0.5% percent difference between teh volume change calculated directly from volume versus the volume change calculated based on the flows across cell edges.

In [107]:
real_cells = transport_model.mesh.nreal

In [135]:
df_ls = []
df_diff = []

for cell in range(real_cells):
    results, diff = build_cell_dataframe(cell, threshold=0.5)
    df_ls.append(results)
    df_diff.append(diff)
    if cell % 5000 == 0:
        print(cell)

0
5000
10000
15000
20000
25000
30000
35000
40000
45000



KeyboardInterrupt



In [136]:
total_df = pd.concat(df_ls)

In [137]:
total_diff = pd.concat(df_diff)

In [138]:
total_df.to_parquet('output_full.parquet')

In [139]:
total_diff.to_parquet('output_pct_diff.parquet')